# 3D U-Net + FADC-Encoder **V2** — Spatial k_att where spatial extent is largest

**Hypothesis being tested.** V2's spatial k_att (per-voxel softmax over dilation branches) underperformed at the Bottleneck variant (0.6648 vs v1's 0.6801) — possibly because the 8×8×4 = 256 voxels at bn give the mechanism almost no spatial room. Encoder placement gives spatial k_att **much** more room to do meaningful per-voxel adaptation:

| Encoder level | Spatial extent (96³×48 input) | k_att map size |
|---|---|---|
| enc1 | 96 × 96 × 48 | ~442,000 voxels |
| enc2 | 48 × 48 × 24 |  ~55,000 voxels |
| enc3 | 24 × 24 × 12 |   ~6,900 voxels |
| enc4 | 12 × 12 × 6  |    ~864 voxels |
| (bn at 6 × 6 × 3 = 108 voxels — but FADC NOT at bn here) | | |

**v1 Encoder reference (uncontrolled seed):** 0.6527 — LOST baseline by −0.0208.
**Baseline 3D U-Net 2ch:** 0.6735.

**Decision logic.**
- v2 Encoder ≥ 0.69 → fix demonstrably matters where spatial extent allows. Run s=123/999.
- v2 Encoder 0.66–0.69 → improvement over v1 Encoder; tells us spatial extent gates the fix's effectiveness. Worth one extra seed.
- v2 Encoder < 0.66 → spatial k_att doesn't help on this dataset regardless of extent. Stop.

**Setup:** FADC at enc1+enc2+enc3+enc4 only (no bn, no decoder). Patch size **96×96×48** (encoder runs at full resolution → tighter memory budget than Bottleneck variant). 100 epochs, batch 2, seed 42, cudnn deterministic.

**GPU:** Kaggle T4 x2. ~6–7h expected wall time.

**IMPORTANT:** Download `best_model.pth`, `train_log.json`, `meta.json` before session closes.

In [ ]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
SEED = 42

DATA_ROOT    = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
OUTPUT_DIR   = f"/kaggle/working/outputs/fadc_encoder_v21_2ch_100ep_s{SEED}"
CODE_DIR     = "/kaggle/working/FADC-3D"

EPOCHS       = 100
BATCH_SIZE   = 2
NUM_WORKERS  = 4
PATCH_SIZE   = [96, 96, 48]   # SMALLER patch — encoder FADC at full resolution
WARMUP       = 5
VAL_EVERY    = 10   # validate every N epochs (config.yaml default is 25)

K_ATT_TEMP_START    = 4.0
K_ATT_TEMP_END      = 0.8
K_ATT_ANNEAL_EPOCHS = 100     # anneal over the full run

RESUME_FROM  = ""
PREPROCESSED_CACHE_DIR = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"

GIT_BRANCH = "feature/fadc-3d-v2-spatial"

print(f"SEED         : {SEED}")
print(f"PATCH_SIZE   : {PATCH_SIZE}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"k_att temp   : {K_ATT_TEMP_START} -> {K_ATT_TEMP_END} over {K_ATT_ANNEAL_EPOCHS} ep")
print(f"val_every    : {VAL_EVERY}")

In [ ]:
# ─────────────────────────────────────────────
# 1. INSTALL DEPENDENCIES
# ─────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "monai",
                "--upgrade-strategy", "only-if-needed", "-q"], check=True)

import torch
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    p = torch.cuda.get_device_properties(0)
    print(f"VRAM           : {p.total_memory / 1e9:.1f} GB")

In [ ]:
# ─────────────────────────────────────────────
# 2. CLONE / UPDATE V2 BRANCH
# ─────────────────────────────────────────────
import os, sys
if os.path.exists(CODE_DIR):
    print("Repo exists — fetching + v2 branch...")
    os.system(f"git -C {CODE_DIR} fetch --all")
    os.system(f"git -C {CODE_DIR} checkout {GIT_BRANCH}")
    os.system(f"git -C {CODE_DIR} pull")
else:
    os.system(f"git clone -b {GIT_BRANCH} https://github.com/Vemuri-BK/FADC-3D.git {CODE_DIR}")
sys.path.insert(0, CODE_DIR)

for p in ["fadc_3d_v2/omni_attention_3d_spatial.py",
         "models/unet_3d_fadc_v2.py",
         "training/train_centralized_v2.py"]:
    assert os.path.exists(os.path.join(CODE_DIR, p)), f"missing: {p}"
print("V2 modules present.")

In [ ]:
# ─────────────────────────────────────────────
# VERIFY TEMPERATURE-SCOPE FIX (commit 476fb8b)
# Aborts if the git pull didn't pick up the fix — saves 7h wasted training.
# ─────────────────────────────────────────────
import inspect, sys
sys.path.insert(0, CODE_DIR)
from fadc_3d_v2.omni_attention_3d_spatial import OmniAttention3DSpatial

ch_src = inspect.getsource(OmniAttention3DSpatial.get_channel_attention)
fi_src = inspect.getsource(OmniAttention3DSpatial.get_filter_attention)
sp_src = inspect.getsource(OmniAttention3DSpatial.get_spatial_attention)
ka_src = inspect.getsource(OmniAttention3DSpatial.get_kernel_attention_spatial)

assert 'self.temperature' not in ch_src, 'BUG STILL PRESENT in get_channel_attention — did the git pull actually run? Expected commit 476fb8b.'
assert 'self.temperature' not in fi_src, 'BUG STILL PRESENT in get_filter_attention.'
assert 'self.temperature' not in sp_src, 'BUG STILL PRESENT in get_spatial_attention.'
assert 'self.temperature' in ka_src,    'k_att lost its temperature term — softmax needs it. Wrong commit?'

print('Temperature-scope fix verified:')
print('  c_att sigmoid : NO temperature term (correct)')
print('  f_att sigmoid : NO temperature term (correct)')
print('  s_att sigmoid : NO temperature term (correct)')
print('  k_att softmax : temperature term present (correct)')
print()
print('Warm-init at bias=0.5 should now produce sigmoid(0.5) = 0.622 (not 0.531).')


In [ ]:
# ─────────────────────────────────────────────
# 3. SMOKE TEST — verify v2 ENCODER instantiates + spatial k_att at all 8 FADC blocks
#    (4 encoder blocks x conv1+conv2 each)
# ─────────────────────────────────────────────
import torch
from models.unet_3d_fadc_v2 import UNet3DFADC_V2
from fadc_3d_v2.adaptive_dilated_conv_3d_v2 import AdaptiveDilatedConv3DV2

device = torch.device("cuda")
model = UNet3DFADC_V2(in_channels=2, out_channels=2, base_filters=32,
                       fadc_placement='encoder').to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Params           : {n_params:,}")
print(f"(v1 Encoder was : ~45,000,000)")

k_att_shapes = {}
for name, m in model.named_modules():
    if isinstance(m, AdaptiveDilatedConv3DV2):
        def mk(nm):
            def hook(_mod, _inp, out):
                k_att_shapes[nm] = tuple(out[3].shape)
            return hook
        m.omni_att.register_forward_hook(mk(name))

x = torch.randn(1, 2, *PATCH_SIZE).to(device)
with torch.no_grad():
    y = model(x)
print(f"Forward OK. Output: {y.shape}\n")

print("Per-block k_att shapes (must have spatial dims > 1):")
for name, s in k_att_shapes.items():
    spatial_size = s[2] * s[3] * s[4]
    print(f"  {name:<32}  shape={s}  spatial_voxels={spatial_size:,}")
    assert s[2] > 1 and s[3] > 1 and s[4] > 1, f"k_att not spatial at {name}"

print(f"\nTotal FADC blocks: {len(k_att_shapes)} (expected 8 = 4 encoder x 2 convs)")
del model, x, y
torch.cuda.empty_cache()
print("V2 ENCODER smoke test PASSED.")

In [ ]:
# ─────────────────────────────────────────────
# 4. CACHE SANITY
# ─────────────────────────────────────────────
import os, numpy as np
from pathlib import Path

cache_path = Path(PREPROCESSED_CACHE_DIR)
assert cache_path.exists(), f"Cache not found: {cache_path}"
train_npzs = sorted((cache_path / "train").glob("*.npz"))
val_npzs   = sorted((cache_path / "val").glob("*.npz"))
print(f"Train : {len(train_npzs)} | Val: {len(val_npzs)}")
for p in [train_npzs[0], train_npzs[-1], val_npzs[0]]:
    d = np.load(p)
    print(f"  {p.name}  image={d['image'].shape}  label={d['label'].shape}")
    assert d['image'].shape[0] == 2, f"NOT 2-channel: {p.name}"
print("Cache OK.")

In [ ]:
# ─────────────────────────────────────────────
# 5. LAUNCH V2 ENCODER TRAINING
# ─────────────────────────────────────────────
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_script = os.path.join(CODE_DIR, "training", "train_centralized_v2.py")

cmd = [
    sys.executable, "-u", train_script,
    "--model",          "unet3d_fadc_encoder_v2",
    "--data_root",      DATA_ROOT,
    "--output_dir",     OUTPUT_DIR,
    "--epochs",         str(EPOCHS),
    "--batch_size",     str(BATCH_SIZE),
    "--num_workers",    str(NUM_WORKERS),
    "--patch_size",     str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--warmup_epochs",  str(WARMUP),
    "--val_every",      str(VAL_EVERY),
    "--seed",           str(SEED),
    "--k_att_temp_start", str(K_ATT_TEMP_START),
    "--k_att_temp_end",   str(K_ATT_TEMP_END),
]
if K_ATT_ANNEAL_EPOCHS is not None:
    cmd += ["--k_att_anneal_epochs", str(K_ATT_ANNEAL_EPOCHS)]
if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM]
if PREPROCESSED_CACHE_DIR:
    cmd += ["--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR]

print("Command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
process.wait()
print(f"\nExit code: {process.returncode}")

In [ ]:
# ─────────────────────────────────────────────
# 6. TRAINING CURVES (Loss + Val Dice + k_att temperature)
# ─────────────────────────────────────────────
import json, os
import matplotlib.pyplot as plt

BASELINE_DICE        = 0.6735
ENCODER_V1_DICE      = 0.6527   # v1 Encoder uncontrolled seed
BOTTLENECK_V2_DICE   = 0.6648   # v2 Bottleneck s=42

log_path = os.path.join(OUTPUT_DIR, "train_log.json")
if not os.path.exists(log_path):
    print("No training log yet.")
else:
    with open(log_path) as f:
        log = json.load(f)
    epochs     = [e["epoch"] for e in log]
    losses     = [e["loss"]  for e in log]
    temps      = [e.get("k_att_temperature", float("nan")) for e in log]
    val_epochs = [e["epoch"]    for e in log if "val_dice" in e]
    val_dices  = [e["val_dice"] for e in log if "val_dice" in e]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(epochs, losses, color="steelblue")
    axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch"); axes[0].grid(True, alpha=0.3)

    axes[1].plot(val_epochs, val_dices, color="darkorange", marker="o", markersize=4)
    axes[1].axhline(BASELINE_DICE,       color="green", ls="--", label=f"Baseline ({BASELINE_DICE:.4f})")
    axes[1].axhline(ENCODER_V1_DICE,     color="red",   ls="--", label=f"v1 Encoder ({ENCODER_V1_DICE:.4f})")
    axes[1].axhline(BOTTLENECK_V2_DICE,  color="purple",ls=":",  label=f"v2 Bottleneck s=42 ({BOTTLENECK_V2_DICE:.4f})")
    axes[1].set_title("Val Dice"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    if val_dices:
        axes[1].set_title(f"Val Dice  (best: {max(val_dices):.4f})")

    axes[2].plot(epochs, temps, color="darkred")
    axes[2].set_title("k_att temperature"); axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("T")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
    plt.show()

    if val_dices:
        best = max(val_dices)
        print(f"Best Val Dice            : {best:.4f}")
        print(f"vs v1 Encoder            : {best - ENCODER_V1_DICE:+.4f}")
        print(f"vs Baseline              : {best - BASELINE_DICE:+.4f}")
        print(f"vs v2 Bottleneck s=42    : {best - BOTTLENECK_V2_DICE:+.4f}")

In [ ]:
# ─────────────────────────────────────────────
# 7. POST-TRAIN k_att DIAGNOSTIC — at each of 8 FADC blocks
#    Reports per-block adaptation evidence: does spatial k_att vary per input?
# ─────────────────────────────────────────────
import os, sys, torch, numpy as np
from pathlib import Path
sys.path.insert(0, CODE_DIR)
from data.mama_mia_dataset import build_centralized_loaders
from models.unet_3d_fadc_v2 import UNet3DFADC_V2
from fadc_3d_v2.adaptive_dilated_conv_3d_v2 import AdaptiveDilatedConv3DV2
from monai.inferers import sliding_window_inference

ckpt_path = os.path.join(OUTPUT_DIR, "best_model.pth")
if not os.path.exists(ckpt_path):
    print("No best_model.pth — run training first.")
else:
    device = torch.device("cuda")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    model = UNet3DFADC_V2(in_channels=cfg["model"]["in_channels"],
                           out_channels=cfg["model"]["out_channels"],
                           base_filters=cfg["model"]["base_filters"],
                           fadc_placement='encoder').to(device).eval()
    model.load_state_dict(ckpt["model"])
    print(f"Loaded v2 Encoder ckpt: best_dice={ckpt['best_dice']:.4f}\n")

    captured = {}
    for name, mod in model.named_modules():
        if isinstance(mod, AdaptiveDilatedConv3DV2):
            captured[name] = []
            def mk(nm):
                def hook(_m, _i, out):
                    k = out[3].detach().cpu().numpy()  # (1, n_branches, d, h, w)
                    captured[nm].append(k[0].mean(axis=(1,2,3)))   # mean across voxels per branch
                return hook
            mod.omni_att.register_forward_hook(mk(name))
    print(f"Hooked {len(captured)} FADC blocks")

    split_csv = os.path.join(DATA_ROOT, "train_test_splits.csv")
    _, val_loader = build_centralized_loaders(
        data_root=DATA_ROOT,
        split_csv=split_csv if os.path.exists(split_csv) else None,
        cache_rate=0.0, num_workers=0, batch_size=1,
        preprocessed_cache_dir=PREPROCESSED_CACHE_DIR,
        patch_size=tuple(cfg["data"]["patch_size"]),
    )
    n = 0
    with torch.no_grad():
        for batch in val_loader:
            if n >= 6: break
            x = batch["image"].to(device)
            _ = sliding_window_inference(x, tuple(cfg["data"]["patch_size"]), 1, model, overlap=0.0)
            n += 1
    print(f"Processed {n} val cases\n")

    print("=" * 86)
    print("V2 ENCODER k_att per-input adaptation diagnostic")
    print("=" * 86)
    print(f"{'block':<34}{'mean k(d=1)':>13}{'std k(d=1)':>13}{'mean k(d=2)':>13}{'adapt?':>10}")
    print("-" * 86)
    for nm in captured:
        arr = np.array(captured[nm])
        if arr.size == 0: continue
        k1m = float(arr[:, 0].mean())
        k1s = float(arr[:, 0].std(ddof=1)) if len(arr) > 1 else 0.0
        k2m = float(arr[:, 1].mean())
        adapts = k1s > 0.005
        print(f"{nm:<34}{k1m:>13.3f}{k1s:>13.4f}{k2m:>13.3f}{'YES' if adapts else 'no':>10}")

    print("\nReference — v2 Bottleneck s=42 (2026-06-25):")
    print("  bottleneck.conv1   std=0.0XXX -> (replace with your result)")
    print("  bottleneck.conv2   std=0.0XXX -> (replace with your result)")

In [ ]:
# ─────────────────────────────────────────────
# 8. DOWNLOAD LINKS
# ─────────────────────────────────────────────
import os
from IPython.display import FileLink, display
for fname in ("best_model.pth", "latest_checkpoint.pth", "train_log.json", "meta.json", "training_curves.png"):
    p = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(p):
        print(fname); display(FileLink(p))
    else:
        print(f"(missing) {fname}")